# Data-Driven Real Estate Investment Decision Model under Uncertainty

## Project roadmap
1. Assumptions and data design
2. ML valuation models and validation
3. Levered cash-flow model
4. Monte Carlo risk (price, rent, rates)
5. Sensitivity and scenario analysis
6. Conclusions


## 1) Assumptions and synthetic data

The dataset is synthetic but structured to preserve realistic economic relationships among property fundamentals, location quality, and macro variables.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.real_estate_investment_model import (
    InvestmentAssumptions,
    build_cashflows,
    create_model_matrix,
    evaluate_models,
    generate_synthetic_real_estate_data,
    irr,
    monte_carlo_investment,
    npv,
    run_sensitivity_grid,
)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120
Path('../results/figures').mkdir(parents=True, exist_ok=True)

In [ ]:
df = generate_synthetic_real_estate_data(n_properties=3000, years=10, seed=42)
X, y, features = create_model_matrix(df)
print('Data shape:', df.shape)
df[['sale_price','monthly_rent','mortgage_rate','gdp_growth']].describe().T

## 2) ML pricing models

We evaluate three models with train/validation/test split and rank by validation RMSE to avoid overfitting to a single holdout.


In [ ]:
metrics, fitted_models, splits = evaluate_models(X, y, random_state=42)
metrics

In [ ]:
best_model_name = metrics.iloc[0]['model']
best_model = fitted_models[best_model_name]
print('Selected model (lowest validation RMSE):', best_model_name)

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=False).head(12)
else:
    fi = pd.Series(np.abs(best_model.coef_), index=features).sort_values(ascending=False).head(12)

fig, ax = plt.subplots(figsize=(10,6))
sns.barplot(x=fi.values, y=fi.index, ax=ax, color='#205375')
ax.set_title(f'Top Feature Importance ({best_model_name})')
ax.set_xlabel('Importance')
fig.tight_layout()
fig.savefig('../results/figures/feature_importance.png', bbox_inches='tight')
plt.show()

## 3) Financial model (levered cash flow)

Cash flows include acquisition outlay, debt service, NOI, and exit proceeds net of sale costs and remaining loan balance.


In [ ]:
assump = InvestmentAssumptions()
price_path = np.full(assump.holding_period_years, assump.annual_price_growth_mu)
rent_path = np.full(assump.holding_period_years, assump.rent_growth_mu)
rate_path = np.full(assump.holding_period_years, assump.annual_interest_rate)

base_cf = build_cashflows(assump, price_path, rent_path, rate_path)
cf = base_cf['cashflows']
pd.Series({
    'NPV': npv(assump.discount_rate, cf),
    'IRR': irr(cf),
    'Payback (years)': base_cf['payback_period'],
    'Terminal Value': base_cf['property_value_terminal'],
    'Net Sale Proceeds': base_cf['sale_proceeds']
}).to_frame('Base Case')

## 4) Monte Carlo simulation (12,000 paths)

Uncertain drivers include annual **price growth**, **rent growth**, and **interest rates** (mean-reverting).


In [ ]:
mc = monte_carlo_investment(assump, n_sims=12000, seed=11)
mc[['irr','npv','avg_interest_rate']].describe(percentiles=[0.01,0.05,0.5,0.95,0.99]).T

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))
sns.histplot(mc['irr'].dropna(), bins=60, kde=True, ax=axes[0], color='#2A9D8F')
axes[0].set_title('IRR Distribution')
sns.histplot(mc['npv'], bins=60, kde=True, ax=axes[1], color='#E9C46A')
axes[1].set_title('NPV Distribution')
sns.histplot(mc['avg_interest_rate'], bins=50, kde=True, ax=axes[2], color='#E76F51')
axes[2].set_title('Average Interest Rate Distribution')
fig.tight_layout()
fig.savefig('../results/figures/return_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
risk_summary = pd.Series({
    'P(NPV<0)': (mc['npv'] < 0).mean(),
    'P(IRR<12%)': (mc['irr'] < 0.12).mean(),
    'NPV VaR 5%': mc['npv'].quantile(0.05),
    'NPV CVaR 5%': mc.loc[mc['npv'] <= mc['npv'].quantile(0.05), 'npv'].mean()
})
risk_summary.to_frame('Value')

## 5) Sensitivity and scenario analysis

In [ ]:
sens = run_sensitivity_grid(assump)
pivot_npv = sens.pivot_table(index='interest_rate', columns='purchase_price', values='npv', aggfunc='mean')
pivot_irr = sens.pivot_table(index='interest_rate', columns='annual_rent', values='irr', aggfunc='mean')

fig, axes = plt.subplots(1,2,figsize=(14,5))
sns.heatmap(pivot_npv, annot=True, fmt='.0f', cmap='RdYlGn', ax=axes[0])
axes[0].set_title('NPV Sensitivity (Rate × Purchase Price)')
sns.heatmap(pivot_irr, annot=True, fmt='.3f', cmap='RdYlGn', ax=axes[1])
axes[1].set_title('IRR Sensitivity (Rate × Rent)')
fig.tight_layout()
fig.savefig('../results/figures/sensitivity_heatmaps.png', bbox_inches='tight')
plt.show()

In [ ]:
scenario_df = pd.DataFrame([
    {'scenario':'Bear','price_mu':0.00,'rent_mu':0.01,'rate':0.068},
    {'scenario':'Base','price_mu':0.03,'rent_mu':0.025,'rate':0.056},
    {'scenario':'Bull','price_mu':0.055,'rent_mu':0.04,'rate':0.048},
])

rows = []
for _, r in scenario_df.iterrows():
    a = InvestmentAssumptions(annual_price_growth_mu=r['price_mu'], rent_growth_mu=r['rent_mu'], annual_interest_rate=r['rate'])
    out = monte_carlo_investment(a, n_sims=6000, seed=101)
    rows.append({'Scenario': r['scenario'], 'Median IRR': out['irr'].median(), 'Median NPV': out['npv'].median(), 'P(NPV<0)': (out['npv']<0).mean()})
scenario_results = pd.DataFrame(rows)
scenario_results

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
tmp = scenario_results.melt(id_vars='Scenario', value_vars=['Median IRR','P(NPV<0)'], var_name='Metric')
sns.barplot(data=tmp, x='Scenario', y='value', hue='Metric', ax=ax)
ax.set_title('Scenario Comparison')
fig.tight_layout()
fig.savefig('../results/figures/scenario_comparison.png', bbox_inches='tight')
plt.show()

## 6) Conclusion
This framework links ML valuation with explicit downside-risk quantification and financing sensitivity, yielding investment decisions based on distributions rather than point estimates.